# Modelagem de demanda — Issue #12

Este notebook reproduz a validação temporal do modelo de previsão. O período futuro não entra no treino nem no cálculo das features.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

RAIZ = Path.cwd().resolve().parents[1] if Path.cwd().name == 'exploracao' else Path.cwd()
sys.path.append(str(RAIZ))

from src.features.pipeline import gerar_features
from src.models.modelo_demanda import (
    avaliar_validacao_temporal,
    prever_demanda,
    salvar_modelo,
    treinar_modelo,
)

dados = pd.read_csv(RAIZ / 'data' / 'processed' / 'consumo_medicamentos.csv')
dados['data'] = pd.to_datetime(dados['data'])
dados.shape, dados['medicamento_id'].nunique(), (dados['data'].min(), dados['data'].max())

## Validação temporal

O corte deixa os últimos sete dias exclusivamente para validação. Não há divisão aleatória.

In [ ]:
corte = dados['data'].max() - pd.Timedelta(days=7)
comparacao = avaliar_validacao_temporal(dados, data_corte=corte)
print(f'Corte: {corte.date()}')
print(f"MAE temporal: {comparacao.attrs['mae']:.2f}")
comparacao.head()

## Treino final e previsão

Após validar, treinamos com todo o histórico disponível e salvamos o artefato localmente. As previsões usam lags, médias móveis, calendário e variáveis externas; `horizonte_dias` permite estimar diretamente cada um dos sete dias futuros.

In [ ]:
features = gerar_features(dados)
modelo = treinar_modelo(features)
salvar_modelo(modelo)

previsoes = prever_demanda(modelo, features, data_corte=dados['data'].max())
previsoes.head(14)